# 02 — Baseline generators for the PLOS tail

Generators: **SMOTE-NC**, **Gaussian copula**, **CTGAN**, **TVAE** (all trained on TRAIN only).
Statistical generators (copula, CTGAN, TVAE) learn from the PLOS rows only; SMOTE-NC interpolates between PLOS neighbours.

Each generator is scored on:
1. **Validity**: share of generated rows that are actually PLOS (LOS > threshold)
2. **Tail LOS fidelity**: Wasserstein / KS vs real PLOS LOS, extreme-tail coverage
3. **Mode coverage**: share of each mode (addiction_pain, cardiopulmonary, oncology_transplant, unexplained) vs real
4. **Condition fidelity**: mean abs. difference in condition prevalence, correlation difference
5. **Copying**: exact copies + distance to closest real record vs a real holdout baseline
6. **Utility**: AUPRC / AUROC on VAL when synthetic PLOS rows are added to train

TEST split is not touched here (kept for final results).


In [ ]:
# Cell 1 — install (Colab)
!pip -q install sdv imbalanced-learn


In [ ]:
# Cell 2 — paths, settings, load
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/genai_synthea/run_01"
OUT  = f"{BASE}/genai"
SYN_DIR = f"{OUT}/synthetic"
RES_DIR = f"{OUT}/results"

SEED     = 42
N_SYN    = 1000      # synthetic PLOS rows kept per generator (for fidelity metrics)
EPOCHS   = 300       # CTGAN / TVAE


In [ ]:
# Cell 3 — load splits + metadata, define columns
import os, json, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
os.makedirs(SYN_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)

meta  = json.load(open(f"{OUT}/meta.json"))
train = pd.read_parquet(f"{OUT}/train.parquet")
val   = pd.read_parquet(f"{OUT}/val.parquet")

THR   = meta["plos_threshold"]
MODES = meta["modes"]
NUM   = meta["num_cols"] + ["target"]          # age + LOS (LOS is generated too)
CAT   = meta["cat_cols"]
BIN   = meta["bin_cols"]
GEN_COLS = NUM + CAT + BIN

real_tail = train[train["is_plos"] == 1][GEN_COLS].reset_index(drop=True)
print(f"Train: {len(train)} rows | PLOS (generator training set): {len(real_tail)}")
print(f"Val:   {len(val)} rows | PLOS: {val['is_plos'].sum()}")
print(f"Columns generated: {len(NUM)} numeric, {len(CAT)} categorical, {len(BIN)} condition flags")


In [ ]:
# Cell 4 — helpers: clean generated output, mode flags
def clean(df):
    df = df[GEN_COLS].copy()
    for c in BIN:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).round().clip(0, 1).astype(int)
    for c in CAT:
        df[c] = df[c].astype(str)
    df["anchor_age"] = pd.to_numeric(df["anchor_age"], errors="coerce").round()
    df["target"] = pd.to_numeric(df["target"], errors="coerce").round()
    return df.dropna(subset=["anchor_age", "target"]).reset_index(drop=True)

def mode_flags(d):
    f = pd.DataFrame({k: d[[c for c in v if c in d.columns]].max(axis=1) for k, v in MODES.items()}, index=d.index)
    f["unexplained"] = (f.sum(axis=1) == 0).astype(int)
    return f

def keep_valid(df, n):
    """Return (first n rows with LOS > THR, validity rate of all generated rows)."""
    valid = df["target"] > THR
    return df[valid].head(n).reset_index(drop=True), float(valid.mean())

synthetic, validity, fit_time = {}, {}, {}


In [ ]:
# Cell 5 — SMOTE-NC (interpolates between PLOS neighbours; uses all train rows for fitting)
from imblearn.over_sampling import SMOTENC

t0 = time.time()
X = train[GEN_COLS].copy()
for c in BIN: X[c] = X[c].astype(str)          # treat flags as categorical so they stay 0/1
y = train["is_plos"].values
cat_idx = [X.columns.get_loc(c) for c in CAT + BIN]
n_pos = int(y.sum())
sm = SMOTENC(categorical_features=cat_idx, sampling_strategy={1: n_pos + N_SYN}, k_neighbors=5, random_state=SEED)
X_res, _ = sm.fit_resample(X, y)
smote = clean(X_res.iloc[len(X):])
synthetic["SMOTE-NC"], validity["SMOTE-NC"] = keep_valid(smote, N_SYN)
fit_time["SMOTE-NC"] = time.time() - t0
print(f"SMOTE-NC: {len(synthetic['SMOTE-NC'])} rows | validity {validity['SMOTE-NC']:.1%} | {fit_time['SMOTE-NC']:.0f}s")


In [ ]:
# Cell 6 — SDV generators trained on PLOS rows only: Gaussian copula, CTGAN, TVAE
import torch
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer, CTGANSynthesizer, TVAESynthesizer

train_tail = real_tail.copy()
for c in BIN: train_tail[c] = train_tail[c].astype(str)   # categorical, so outputs stay 0/1

md_ = Metadata.detect_from_dataframe(train_tail, table_name="plos")
for c in CAT + BIN: md_.update_column(column_name=c, sdtype="categorical", table_name="plos")
for c in NUM:       md_.update_column(column_name=c, sdtype="numerical", table_name="plos")

def make(name):
    if name == "GaussianCopula":
        return GaussianCopulaSynthesizer(md_, enforce_min_max_values=True)
    if name == "CTGAN":
        return CTGANSynthesizer(md_, epochs=EPOCHS, batch_size=100, enforce_min_max_values=True, verbose=False)
    if name == "TVAE":
        return TVAESynthesizer(md_, epochs=EPOCHS, batch_size=100, enforce_min_max_values=True)

for name in ["GaussianCopula", "CTGAN", "TVAE"]:
    np.random.seed(SEED); torch.manual_seed(SEED)
    t0 = time.time()
    model = make(name)
    model.fit(train_tail)
    raw = clean(model.sample(num_rows=3 * N_SYN))
    synthetic[name], validity[name] = keep_valid(raw, N_SYN)
    fit_time[name] = time.time() - t0
    print(f"{name:15s}: {len(synthetic[name])} valid rows | validity {validity[name]:.1%} | {fit_time[name]:.0f}s")

for name, df in synthetic.items():
    df.to_parquet(f"{SYN_DIR}/{name}.parquet", index=False)


In [ ]:
# Cell 7 — fidelity, mode coverage, copying
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.neighbors import NearestNeighbors

real_flags = mode_flags(real_tail).mean()
real_p90   = real_tail["target"].quantile(0.90)   # extreme tail inside the PLOS tail

# encoding for distance computations (fit on real tail)
age_mu, age_sd = real_tail["anchor_age"].mean(), real_tail["anchor_age"].std()
los_mu, los_sd = real_tail["target"].mean(), real_tail["target"].std()
cat_levels = {c: sorted(real_tail[c].astype(str).unique()) for c in CAT}
def encode(d):
    parts = [((d["anchor_age"] - age_mu) / age_sd).to_frame(), ((d["target"] - los_mu) / los_sd).to_frame(), d[BIN].astype(float)]
    for c in CAT:
        parts.append(pd.DataFrame({f"{c}={l}": (d[c].astype(str) == l).astype(float) for l in cat_levels[c]}, index=d.index))
    return pd.concat(parts, axis=1).values

nn = NearestNeighbors(n_neighbors=1).fit(encode(real_tail))
holdout = val[val["is_plos"] == 1][GEN_COLS].reset_index(drop=True)
dcr_holdout = np.median(nn.kneighbors(encode(holdout))[0])
real_corr = real_tail[["anchor_age", "target"] + BIN].corr().fillna(0).values

rows = []
for name, s in synthetic.items():
    f = mode_flags(s).mean()
    d_syn = nn.kneighbors(encode(s))[0].ravel()
    corr_diff = np.abs(s[["anchor_age", "target"] + BIN].corr().fillna(0).values - real_corr)
    rows.append({
        "generator": name,
        "validity": validity[name],
        "LOS_wasserstein": wasserstein_distance(real_tail["target"], s["target"]),
        "LOS_KS": ks_2samp(real_tail["target"], s["target"]).statistic,
        "extreme_cov": (s["target"] > real_p90).mean(),           # real value is ~0.10
        "LOS_max": s["target"].max(),
        **{f"mode_{k}": f[k] for k in f.index},
        "mode_abs_err": (f - real_flags).abs().mean(),
        "cond_prev_err": (s[BIN].mean() - real_tail[BIN].mean()).abs().mean(),
        "corr_err": corr_diff[np.triu_indices_from(corr_diff, 1)].mean(),
        "exact_copies": (d_syn < 1e-6).mean(),
        "DCR_ratio": np.median(d_syn) / dcr_holdout,               # <1 = closer to train than real unseen data
        "fit_s": fit_time[name],
    })

real_row = {"generator": "REAL (train PLOS)", "validity": 1.0, "LOS_wasserstein": 0, "LOS_KS": 0,
            "extreme_cov": (real_tail["target"] > real_p90).mean(), "LOS_max": real_tail["target"].max(),
            **{f"mode_{k}": real_flags[k] for k in real_flags.index}}
fid = pd.DataFrame([real_row] + rows).set_index("generator")
fid.to_csv(f"{RES_DIR}/02_fidelity.csv")
print(f"Real PLOS: {len(real_tail)} rows | LOS p90 inside tail = {real_p90:.0f} d | holdout median DCR = {dcr_holdout:.2f}")
fid.round(3).T


In [ ]:
# Cell 8 — utility: add synthetic PLOS rows to train, evaluate on VAL
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

FEAT_NUM, FEAT = ["anchor_age"], None
def X_of(d):
    x = pd.get_dummies(d[FEAT_NUM + CAT + BIN], columns=CAT, dtype=float)
    return x

base_X = X_of(train); FEAT = base_X.columns
Xv = X_of(val).reindex(columns=FEAT, fill_value=0); yv = val["is_plos"].values
n_neg, n_pos = int((train.is_plos == 0).sum()), int(train.is_plos.sum())
N_AUG = max(n_neg // 2 - n_pos, 0)      # bring positives up to a 1:2 ratio
print(f"Adding {N_AUG} synthetic PLOS rows per generator (train: {n_pos} pos / {n_neg} neg)")

def fit_eval(X, y, weight=None):
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED, class_weight=weight)
    clf.fit(X, y)
    p = clf.predict_proba(Xv)[:, 1]
    return average_precision_score(yv, p), roc_auc_score(yv, p)

util = []
ap, auc = fit_eval(base_X, train["is_plos"].values)
util.append({"setting": "Real only", "AUPRC": ap, "AUROC": auc})
ap, auc = fit_eval(base_X, train["is_plos"].values, weight="balanced")
util.append({"setting": "Real only + class weights", "AUPRC": ap, "AUROC": auc})

for name, s in synthetic.items():
    if len(s) < N_AUG:   # top up from the saved generator output if needed
        print(f"  note: {name} has only {len(s)} valid rows; using all of them")
    add = s.head(N_AUG).copy()
    Xa = pd.concat([base_X, X_of(add).reindex(columns=FEAT, fill_value=0)], ignore_index=True)
    ya = np.concatenate([train["is_plos"].values, np.ones(len(add), dtype=int)])
    ap, auc = fit_eval(Xa, ya)
    util.append({"setting": f"+ {name} ({len(add)} rows)", "AUPRC": ap, "AUROC": auc})

util = pd.DataFrame(util).set_index("setting")
util.to_csv(f"{RES_DIR}/02_utility.csv")
print(f"VAL PLOS prevalence (AUPRC of a random model): {yv.mean():.3f}")
util.round(3)


In [ ]:
# Cell 9 — plots: LOS distribution and mode coverage
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
bins = np.arange(THR, max(real_tail["target"].max(), max(s["target"].max() for s in synthetic.values())) + 2, 2)
axes[0].hist(real_tail["target"], bins=bins, density=True, histtype="step", lw=2.5, color="black", label="REAL")
for name, s in synthetic.items():
    axes[0].hist(s["target"], bins=bins, density=True, histtype="step", lw=1.3, label=name)
axes[0].set_yscale("log"); axes[0].set_xlabel("LOS (days)"); axes[0].set_title("PLOS tail: LOS"); axes[0].legend()

mode_cols = [c for c in fid.columns if c.startswith("mode_") and c != "mode_abs_err"]
fid[mode_cols].rename(columns=lambda c: c.replace("mode_", "")).T.plot.bar(ax=axes[1], rot=0)
axes[1].set_ylabel("share of PLOS rows"); axes[1].set_title("Mode coverage (REAL vs generators)")
fig.tight_layout(); fig.savefig(f"{RES_DIR}/02_plots.png", dpi=150); plt.show()
print("Saved results to", RES_DIR)
